---
title: "Customer Winback Model"
subtitle: "Predicting Organic Customer Reactivation"
author: "Marcial Palacios"
date: "2026-01-30"
format:
  html:
    toc: true
    toc-depth: 3
    code-fold: true
    code-tools: true
    theme: cosmo
jupyter: python3
---

# Project Overview

This notebook documents the development and evaluation of a **customer winback propensity model** using historical customer information.

The objective was to identify inactive customers who were more likely to return to an active relationship with the financial institution.

The modeling approach combines:

- Weight of Evidence (WOE)
- Information Value (IV)
- Logistic regression
- ROC / AUC analysis
- KS statistic
- Gini coefficient
- Lift and Gain analysis
- Confusion matrices
- DeLong's test for correlated ROC curves

> **Important:** This is a propensity model. The model predicts the likelihood that an inactive customer will reactivate. It does not estimate whether a customer will reactivate *because of* a specific intervention.

# 1. Business Problem

Financial institutions have large populations of customers who become inactive over time.

However, not all inactive customers have the same probability of returning. Some customers may naturally re-establish an active relationship, while others may be unlikely to do so without additional intervention.

The business problem was therefore framed as a **customer ranking problem**:

> Which inactive customers are more likely to return to an active relationship?

The goal was not initially to estimate the causal effect of a campaign, but to develop a predictive model capable of ranking inactive customers according to their probability of reactivation.

This distinction is important because the observed outcome represents **organic customer behavior** rather than the response to a model-driven intervention.

# 2. Modeling Objective

The target variable was defined as:

$$P(Y=1 \mid X)$$

where:

- $Y=1$ indicates that the customer returned to an active relationship.
- $Y=0$ indicates that the customer remained inactive.
- $X$ represents customer characteristics and historical relationship variables available before the outcome was observed.

The final model was designed primarily as a **ranking and segmentation tool**.

# 3. Population and Target Definition

The initial population contained:

**1,539,498 previously inactive customers.**

The binary target was defined as follows:

| Target | Definition                                  |
|-------:|---------------------------------------------|
|      1 | Customer returned to an active relationship |
|      0 | Customer remained inactive                  |

The problem was therefore treated as a binary classification task.

# 4. Data Preparation

The original customer-level data cannot be publicly disclosed because it belongs to the financial institution.

The data preparation and data-cleaning steps are therefore omitted from this public portfolio.

The modeling dataset contained customer relationship variables such as:

- Geographic or commercial location
- Occupation
- Savings relationship
- Checking account relationship
- Employment seniority
- Months inactive
- Credit-related interactions
- Term deposits / investments
- Credit score
- Age
- Customer seniority

The variables used in the model were renamed for confidentiality and reproducibility of the analytical workflow without exposing proprietary product or business terminology.

# 5. Train-Test Partition

The dataset was divided into training and testing samples.

The training dataset was used to:

1.  Calculate WOE transformations.
2.  Estimate the logistic regression coefficients.
3.  Evaluate variable relationships.

The test dataset was kept separate and was only transformed using the WOE mappings learned from the training data.

This separation is important because calculating transformations using the complete dataset could introduce information from the test sample into model development.

```{r}
#| label: setup

library(tidyverse)
library(haven)
library(scorecard)
library(lubridate)
library(arrow)
library(dbplyr)
library(pROC)
library(car)

# Reproducibility
set.seed(123)

# The original data preparation and cleaning steps are omitted.
# df_winback is assumed to contain the final modeling dataset.

data_tt <- split_df(
  df_winback,
  y = "winback"
)

dim(data_tt[["train"]])
dim(data_tt[["test"]])
```

# 6. Weight of Evidence Transformation

Weight of Evidence (WOE) was used to transform the explanatory variables before logistic regression.

For a given variable bin:

$$WOE_i =
\ln
\left(
\frac{\%Good_i}
{\%Bad_i}
\right)$$

WOE provides two useful properties for this analysis:

1.  It helps identify the relationship between predictor values and the target.
2.  It transforms variables into a representation commonly used in credit-risk and scorecard-style logistic regression models.

Information Value (IV) was also considered during variable evaluation.

## 6.1 Candidate Variables

The initial candidate set contained 11 predictors:

```{r}
independent_variables <- c(
  "district",
  "occupation",
  "savings",
  "checking_account",
  "seniority",
  "recency",
  "credit_application",
  "certificate_deposit",
  "credit_score",
  "age",
  "customer_tenure"
)

independent_variables
```

## 6.2 WOE Binning

WOE bins were calculated using **only the training dataset**.

```{r}
woe_wb <- woebin(
  data_tt[["train"]],
  y = "winback",
  x = independent_variables
)
```

The resulting bins can be inspected using the WOE plots:

```{r}
#| fig-width: 8
#| fig-height: 5

map(
  woe_wb,
  woebin_plot,
  line_value = "woe"
)
```

These plots provide a visual representation of how the relationship between each predictor and the winback outcome changes across its bins.

## 6.3 Applying WOE Transformations

After the WOE mappings were calculated using the training data, they were applied to the training sample.

```{r}
woe_wb_data <- woebin_ply(
  data_tt[["train"]],
  woe_wb
)

glimpse(woe_wb_data)
```

The same transformation mappings were subsequently applied to the test sample:

```{r}
data_test <- woebin_ply(
  data_tt[["test"]],
  woe_wb
)
```

This workflow preserves the temporal and modeling separation between training and testing data and avoids using test observations to estimate the WOE transformation.

# 7. Logistic Regression Models

Three model specifications were evaluated.

### Full model

The full model contained all 11 candidate predictors.

### Reduced model

The reduced model retained 9 predictors.

### Two-variable model

A highly simplified specification using only age and customer tenure was also evaluated as a benchmark.

This comparison allowed the analysis to assess whether additional variables provided meaningful incremental predictive performance.

# 8. Full Model

The full logistic regression model was specified as:

```{r}
full_model_wb <- glm(
  winback ~
    district_woe +
    occupation_woe +
    savings_woe +
    checking_account_woe +
    seniority_woe +
    recency_woe +
    credit_application_woe +
    certificate_deposit_woe +
    credit_score_woe +
    age_woe +
    customer_tenure_woe,
  family = binomial(),
  data = woe_wb_data
)

summary(full_model_wb)
```

The model estimates:

$$P(Y=1|X)=
\frac{1}
{1+e^{-(\beta_0+\beta_1X_1+\cdots+\beta_kX_k)}}$$

Because the predictors are WOE-transformed, the coefficients describe the relationship between the transformed predictor values and the log-odds of customer winback.

## 8.1 Multicollinearity

Variance Inflation Factors (VIF) were examined as an additional diagnostic.

```{r}
vif(full_model_wb)
```

# 9. Reduced Model

The reduced model removed two predictors while retaining the remaining nine variables.

```{r}
reduced_model_wb <- glm(
  winback ~
    district_woe +
    occupation_woe +
    savings_woe +
    checking_account_woe +
    seniority_woe +
    recency_woe +
    credit_application_woe +
    certificate_deposit_woe +
    credit_score_woe,
  family = binomial(),
  data = woe_wb_data
)

summary(reduced_model_wb)
```

VIF was again examined:

```{r}
vif(reduced_model_wb)
```

The reduced model became the preferred specification because its predictive performance was virtually identical to that of the full model while requiring fewer predictors.

# 10. Two-Variable Benchmark Model

A second reduced specification was evaluated using only:

- Age
- Customer tenure

```{r}
reduced_2var_model_wb <- glm(
  winback ~
    age_woe +
    customer_tenure_woe,
  family = binomial(),
  data = woe_wb_data
)

summary(reduced_2var_model_wb)
```

This model provides a useful benchmark for evaluating how much discrimination is obtained from a substantially more limited predictor set.

# 11. Probability Estimation

The fitted models were used to estimate the probability of winback for each customer in the test dataset.

```{r}
p_full <- predict(
  full_model_wb,
  newdata = data_test,
  type = "response"
)

p_reduced <- predict(
  reduced_model_wb,
  newdata = data_test,
  type = "response"
)

p_reduced_2var <- predict(
  reduced_2var_model_wb,
  newdata = data_test,
  type = "response"
)
```

Each customer therefore receives an estimated probability:

$$\hat{P}(Winback=1|X)$$

These probabilities can be used to rank customers from highest to lowest predicted propensity.

# 12. Model Evaluation

The models were evaluated on the **test dataset**, which was not used to estimate the logistic regression coefficients.

The evaluation included:

- ROC curve
- AUC
- KS statistic
- Gini coefficient
- Lift
- Gain
- Confusion matrix

```{r}
eval_full <- perf_eva(
  pred = p_full,
  label = data_test$winback,
  title = "Full model",
  binomial_metric = c("ks", "auc", "gini"),
  show_plot = c("ks", "roc", "lift", "gain"),
  confusion_matrix = TRUE
)

eval_reduced <- perf_eva(
  pred = p_reduced,
  label = data_test$winback,
  title = "Reduced model",
  binomial_metric = c("ks", "auc", "gini"),
  show_plot = c("ks", "roc", "lift", "gain"),
  confusion_matrix = TRUE
)

eval_reduced_2var <- perf_eva(
  pred = p_reduced_2var,
  label = data_test$winback,
  title = "Two-variable model",
  binomial_metric = c("ks", "auc", "gini"),
  show_plot = c("ks", "roc", "lift", "gain"),
  confusion_matrix = TRUE
)
```

# 13. ROC Curves

ROC curves were calculated independently using the test observations.

```{r}
roc_full <- roc(
  data_test$winback,
  p_full
)

roc_reduced <- roc(
  data_test$winback,
  p_reduced
)

roc_reduced_2var <- roc(
  data_test$winback,
  p_reduced_2var
)
```

The resulting ROC curves can be compared visually:

```{r}
#| fig-width: 8
#| fig-height: 6

plot(
  roc_full,
  main = "ROC Curve Comparison"
)

lines(
  roc_reduced,
  lty = 2
)

lines(
  roc_reduced_2var,
  lty = 3
)

legend(
  "bottomright",
  legend = c(
    "Full model",
    "Reduced model",
    "Two-variable model"
  ),
  lty = c(1, 2, 3)
)
```

# 14. AUC Results

The main results obtained on the test sample were:

| Model             | Predictors |          AUC |         KS |
|-------------------|-----------:|-------------:|-----------:|
| Full model        |         11 |     0.836444 |     0.5232 |
| **Reduced model** |      **9** | **0.836372** | **0.5230** |

The full model achieved an AUC of **0.836444**, while the reduced model achieved an AUC of **0.836372**.

The absolute difference was:

$$0.836444 - 0.836372 = 0.000072$$

Therefore, although the full model contains two additional predictors, its improvement in discrimination is extremely small.

# 15. DeLong's Test

Because the full and reduced models were evaluated on the **same test observations**, their ROC curves are correlated.

DeLong's test was therefore used to formally compare their AUCs.

```{r}
delong_test <- roc.test(
  roc_full,
  roc_reduced,
  method = "delong"
)

delong_test
```

The results were:

| Statistic         |    Value |
|-------------------|---------:|
| Full model AUC    | 0.836444 |
| Reduced model AUC | 0.836372 |
| AUC difference    | 0.000072 |
| DeLong Z          |   2.7628 |
| p-value           |  0.00573 |

The null hypothesis of equal AUCs is rejected at conventional significance levels.

However, statistical significance must be distinguished from practical significance.

The estimated improvement is only:

**0.000072 AUC points.**

Therefore, although the difference is statistically detectable given the large sample size, the magnitude of the improvement is negligible from a practical modeling perspective.

This supports selecting the **9-variable reduced model** as the final specification.

# 16. Final Model Selection

The reduced model was selected as the final model based on the following considerations:

1.  Its AUC was virtually identical to the full model.
2.  Its KS statistic was also virtually identical.
3.  The full model's incremental AUC was only 0.000072.
4.  The reduced model requires fewer predictors.
5.  Fewer predictors simplify model interpretation and future implementation.
6.  The reduction does not result in a meaningful loss of discrimination.

This illustrates an important modeling principle:

> A statistically significant difference does not necessarily represent a practically meaningful improvement.

In this case, model parsimony was preferred because the additional complexity of the full specification did not provide a meaningful increase in predictive performance.

# 17. Organic Winback Results

The model was subsequently used to identify a smaller segment of inactive customers with a higher predicted propensity to reactivate.

The original inactive population contained:

**1,539,498 customers.**

The resulting activable segment contained:

**194,307 customers.**

This represents:

$$\frac{194,307}{1,539,498}
\approx 12.62\%$$

of the original inactive population.

Among the customers in this segment, the observed reactivation rate was:

**44.02%**

These results demonstrate that the model successfully concentrated customers with a substantially higher observed rate of subsequent reactivation into a relatively small portion of the inactive population.

# 18. Important Interpretation: Propensity vs. Causality

The 44.02% reactivation rate must be interpreted carefully.

The model identified customers who were more likely to return, but the customers classified as activable did **not** receive a targeted intervention as a consequence of the model.

Therefore:

$$44.02\% \neq \text{incremental campaign lift}$$

The observed rate represents **organic customer reactivation**.

The model answers:

> **Which inactive customers are more likely to return?**

It does not answer:

> **Which customers will return because we contact them?**

This distinction is fundamental.

A customer may have a high predicted propensity because of characteristics that are associated with natural reactivation. Contacting that customer may have no additional effect.

Therefore, the model should initially be interpreted as a **propensity-ranking model**, not as a treatment-effect model.

# 19. Business Value

The model provides a mechanism for prioritizing a large inactive customer population.

Instead of treating all 1.5+ million inactive customers equally, the model identifies a smaller population with a higher observed propensity to reactivate.

Potential applications include:

- Winback campaign prioritization
- Customer segmentation
- Contact prioritization
- Resource allocation
- Identification of reactivation opportunities
- Ranking customers according to predicted probability

The most important opportunity is to use the propensity model as a **targeting layer for future experimentation**.

For example, the high-propensity population could be divided into treatment and control groups.

The treatment group would receive a winback intervention, while the control group would not.

The difference in observed reactivation rates would then provide an estimate of incremental impact.

# 20. From Propensity Modeling to Uplift Modeling

A propensity model estimates:

$$P(Y=1|X)$$

This answers:

> Who is likely to reactivate?

However, the business question for a future campaign is different.

The institution may want to know:

> Who is likely to reactivate **because of the intervention**?

This requires experimental or causal methodology.

A randomized experiment could define:

- **Treatment:** customer receives the winback intervention.
- **Control:** customer does not receive the intervention.

The incremental effect could then be estimated as:

$$\Delta =
P(Y=1|Treatment)
-
P(Y=1|Control)$$

The objective would be to identify customers for whom:

$$\Delta > 0$$

and ideally customers for whom the incremental effect is sufficiently large to justify the cost of intervention.

This represents the transition from **propensity modeling** toward **uplift modeling** and causal decision-making.

# 21. Limitations

## No Controlled Intervention

The model was not evaluated through a randomized treatment-control experiment.

Consequently, the analysis cannot estimate the causal effect of a winback campaign.

## Organic Reactivation

The observed 44.02% reactivation rate reflects natural customer behavior.

It should not be interpreted as incremental lift generated by the model.

## Temporal Dependence

The predictors must represent information available before the outcome period.

Future implementations should preserve this temporal structure to prevent data leakage.

## Model Generalization

The model was developed using historical data from a specific customer population and observation period.

Performance should therefore be monitored if the model is applied to a different time period or customer population.

# 22. Key Findings

The analysis produced the following main findings:

- The initial population contained **1,539,498 inactive customers**.
- Eleven candidate predictors were evaluated.
- WOE transformations were calculated using the training data.
- Logistic regression was used as the primary predictive model.
- A full 11-variable model and a reduced 9-variable model were compared.
- The full model achieved an AUC of **0.836444**.
- The reduced model achieved an AUC of **0.836372**.
- The absolute AUC difference was only **0.000072**.
- DeLong's test produced a p-value of **0.00573**.
- Despite statistical significance, the practical difference between the models was negligible.
- The **9-variable reduced model** was therefore selected as the final specification.
- The model identified **194,307 customers**, representing **12.62%** of the original inactive population, as the activable segment.
- The observed reactivation rate within this segment was **44.02%**.
- This reactivation was organic and should not be interpreted as causal campaign lift.
- A controlled experiment would be required to estimate incremental campaign impact.
- Uplift modeling represents a natural next step beyond propensity modeling.

# 23. Reproducibility

The analytical workflow follows this general sequence:

``` text
Historical customer data
          |
          v
   Data preparation
          |
          v
     Train / Test
       split
          |
          v
    WOE binning
    using Train
          |
          v
   WOE transformation
          |
          v
 Logistic Regression
          |
          +------------------+
          |                  |
          v                  v
     Full Model        Reduced Model
      11 vars             9 vars
          |                  |
          +--------+---------+
                   |
                   v
             Test dataset
                   |
                   v
          Probability estimates
                   |
                   v
       ROC / AUC / KS / Gini
                   |
                   v
             Model selection
                   |
                   v
       Customer propensity ranking
```

The original customer-level data is not included in this repository because it contains confidential information belonging to the financial institution.

# 24. Technologies

The analysis was developed using:

- R
- `tidyverse`
- `scorecard`
- `lubridate`
- `arrow`
- `dbplyr`
- `pROC`
- `car`
- Logistic Regression
- Weight of Evidence
- Information Value
- ROC / AUC
- KS
- Gini
- Lift and Gain
- DeLong's test

# 25. Conclusion

This project demonstrates the development of a customer winback propensity model using WOE transformation and logistic regression.

The model achieved strong discrimination, with an AUC of approximately **0.836**, and successfully concentrated customers with a higher observed rate of natural reactivation into a relatively small segment.

The comparison between the 11-variable and 9-variable specifications showed that additional predictors produced a statistically significant but practically negligible improvement in AUC.

The final 9-variable model therefore provides a more parsimonious alternative while preserving virtually the same predictive performance.

Most importantly, the project distinguishes **prediction from causation**.

The model identifies customers who are more likely to reactivate organically. It does not establish that contacting those customers would cause them to reactivate.

The natural next step would therefore be to combine the propensity model with controlled experimentation and, ultimately, uplift modeling to identify customers whose behavior can actually be changed by an intervention.